In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [2]:
# Source sentences (English) and Target sentences (French)
raw_data = [
    ("go", "va"),
    ("run", "cours"),
    ("hi", "salut"),
    ("stop", "arrete"),
    ("wait", "attends"),
    ("see you", "a bientot"),
    ("i try", "j essaye"),
    ("i won", "j ai gagne")
]

In [3]:
# Add start (<start>) and end (<end>) tokens to target sentences
source_text = [pair[0] for pair in raw_data]
target_text_in = [f"<start> {pair[1]} "for pair in raw_data]
target_text_out = [f"{pair[1]} <end> "for pair in raw_data]

In [4]:
# Vectorize English (Source)
src_vector = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=4)
src_vector.adapt(source_text)

In [5]:
# Vectorize French (Target)
text_vector = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=5)
text_vector.adapt(target_text_in + target_text_out)

In [6]:
source_vocab_size = len(src_vector.get_vocabulary())
text_vocab_size = len(text_vector.get_vocabulary())

In [7]:
# Prepare numeric arrays
enc_input_data = src_vector(np.array(source_text)).numpy()
dec_input_data = text_vector(np.array(target_text_in)).numpy()
dec_target_data = text_vector(np.array(target_text_out)).numpy()

In [8]:
# Model Architecture

latent_dim = 32

# --- 1. ENCODER ---

encoder_inputs = tf.keras.layers.Input(shape=(4,), name="enc_in")

enc_emb = tf.keras.layers.Embedding(source_vocab_size, 16, mask_zero=True)(encoder_inputs)

encoder_lstm = tf.keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True, name="enc_lstm")

encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

encoder_states = [state_h, state_c]

In [9]:
# --- 2. DECODER ---

decoder_inputs = tf.keras.layers.Input(shape=(5,),name="dec_in")

dec_emb_layer = tf.keras.layers.Embedding(text_vocab_size, 16, mask_zero=True)

dec_emb = dec_emb_layer(decoder_inputs)

decoder_lstm = tf.keras.layers.LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True,
    name="dec_lstm"
)

decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)

In [10]:
# --- 3. ATTENTION MECHANISM ---
# Query: decoder_outputs (what decoder is thinking right now)
# Value: encoder_outputs (all words the encoder read)

attention_layer = tf.keras.layers.Attention(name="attention_layer")

context_vector = attention_layer([decoder_outputs, encoder_outputs])

# Combine the attention context vector with decoder outputs
decoder_combine_context = tf.keras.layers.Concatenate(axis=-1)([decoder_outputs, context_vector])

In [11]:
# Dense projection
decoder_dense = tf.keras.layers.Dense(text_vocab_size, activation="softmax", name="dec_dense")
final_output = decoder_dense(decoder_combine_context)

In [12]:
# --- COMPILE & TRAIN ---

attn_model = tf.keras.Model([encoder_inputs, decoder_inputs],final_output)

attn_model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.01),
    loss = "sparse_categorical_crossentropy",
    metrics=['accuracy']
)

In [13]:
attn_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ enc_in (InputLayer)           │ (None, 4)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dec_in (InputLayer)           │ (None, 5)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 4, 16)             │             192 │ enc_in[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, 4)                 │               0 │ enc_in[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_1 (Embedding)       │ (None, 5, 16)             │             240 │ dec_in[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ enc_lstm (LSTM)               │ [(None, 4, 32), (None,    │           6,272 │ embedding[0][0],           │
│                               │ 32), (None, 32)]          │                 │ not_equal[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dec_lstm (LSTM)               │ [(None, 5, 32), (None,    │           6,272 │ embedding_1[0][0],         │
│                               │ 32), (None, 32)]          │                 │ enc_lstm[0][1],            │
│                               │                           │                 │ enc_lstm[0][2]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention_layer (Attention)   │ (None, 5, 32)             │               0 │ dec_lstm[0][0],            │
│                               │                           │                 │ enc_lstm[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate (Concatenate)     │ (None, 5, 64)             │               0 │ dec_lstm[0][0],            │
│                               │                           │                 │ attention_layer[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dec_dense (Dense)             │ (None, 5, 15)             │             975 │ concatenate[0][0]          │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 13,951 (54.50 KB)

 Trainable params: 13,951 (54.50 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = attn_model.fit(
    [enc_input_data, dec_input_data],
    dec_target_data,
    epochs=120,
    verbose=1
)

Epoch 1/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.1000 - loss: 2.7095
Epoch 2/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.4000 - loss: 2.6757
Epoch 3/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.4000 - loss: 2.6353
Epoch 4/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.4000 - loss: 2.5816
Epoch 5/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4000 - loss: 2.5080
Epoch 6/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.4000 - loss: 2.4085
Epoch 7/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.4000 - loss: 2.2806
Epoch 8/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4000 - loss: 2.1385
Epoch 9/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.4000 - loss: 2.0285
Epoch 10/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.4000 - loss: 1.9910
Epoch 11/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.4000 - loss: 1.9870
Epoch 12/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.4000 - loss

In [15]:
# 1. Inference Encoder: Outputs all timesteps AND the final states
inf_encoder = tf.keras.Model(
    encoder_inputs, 
    [encoder_outputs, state_h, state_c]
)

# 2. Inference Decoder: Takes 1 token, rolling states, and full encoder outputs
inf_dec_input = tf.keras.layers.Input(shape=(1,))
inf_enc_outputs_input = tf.keras.layers.Input(shape=(4, latent_dim))
inf_state_h = tf.keras.layers.Input(shape=(latent_dim,))
inf_state_c = tf.keras.layers.Input(shape=(latent_dim,))

# Embed single token
inf_emb = dec_emb_layer(inf_dec_input)

# Step decoder LSTM
inf_dec_out, next_h, next_c = decoder_lstm(
    inf_emb, 
    initial_state=[inf_state_h, inf_state_c]
)

# Compute attention for this single step against all encoder timesteps
inf_context = attention_layer([inf_dec_out, inf_enc_outputs_input])

# Concatenate and project to vocabulary
inf_combined = tf.keras.layers.Concatenate(axis=-1)([inf_dec_out, inf_context])
inf_predictions = decoder_dense(inf_combined)

inf_decoder = tf.keras.Model(
    [inf_dec_input, inf_enc_outputs_input, inf_state_h, inf_state_c],
    [inf_predictions, next_h, next_c]
)

In [16]:
# Reconstruct vocabulary lookup maps
tgt_vocab = text_vector.get_vocabulary()
tgt_word_to_idx = {word: i for i, word in enumerate(tgt_vocab)}
tgt_idx_to_word = {i: word for i, word in enumerate(tgt_vocab)}

start_token_idx = tgt_word_to_idx["start"]
end_token_idx = tgt_word_to_idx["end"]

In [20]:
def translate_with_attention(input_sentence):
    # Vectorize input English sentence
    input_seq = src_vector(np.array([input_sentence])).numpy()
    
    # 1. Get all encoder outputs and final states
    enc_outs, h, c = inf_encoder.predict(input_seq, verbose=0)
    
    # Start target sequence with '<start>' token
    target_seq = np.array([[start_token_idx]])
    decoded_words = []
    
    for _ in range(6):
        # Step through decoder with attention over enc_outs
        preds, h, c = inf_decoder.predict(
            [target_seq, enc_outs, h, c], 
            verbose=0
        )
        
        sampled_idx = np.argmax(preds[0, -1, :])
        sampled_word = tgt_idx_to_word[sampled_idx]
        
        if sampled_word == "end":
            break
            
        decoded_words.append(sampled_word)
        target_seq = np.array([[sampled_idx]])
        
    return " ".join(decoded_words)

In [21]:
for phrase in ["go", "run", "hi", "stop", "see you", "i won"]:
    print(f"EN: '{phrase}' --> FR: '{translate_with_attention(phrase)}'")

EN: 'go' --> FR: 'va'
EN: 'run' --> FR: 'cours'
EN: 'hi' --> FR: 'salut'
EN: 'stop' --> FR: 'arrete'
EN: 'see you' --> FR: 'a bientot'
EN: 'i won' --> FR: 'j ai gagne'
